# Clase 12: Detección de Anomalías y Visualización en Baja Dimensionalidad

**MDS7202: Laboratorio de Programación Científica para Ciencia de Datos**




## Objetivos de la Clase


- Distinguir los tipos de anomalías (univariadas y multivariadas) y sus posibles orígenes.
- Detectar outliers univariados usando **Z-score** e **IQR**, y comprender las suposiciones de cada método.
- Reducir la dimensionalidad de un dataset para su visualización con **PCA**, **t-SNE**, **UMAP** y **PaCMAP**, conociendo los hiperparámetros clave de cada uno.
- Detectar anomalías multivariadas con métodos basados en densidad, aislamiento e histogramas: **DBSCAN**, **ECOD**, **IForest**, **LOF**, **HBOS** y **KNN** (vía PyOD).
- Visualizar y comparar los resultados de distintos detectores sobre las proyecciones en baja dimensionalidad.
- Distinguir entre **detección de outliers** y **novelty detection**, y aplicar un detector entrenado a nuevas observaciones.

## Detección y manejo de anomalías

> **Pregunta**: ¿Qué es una anomalía?

Una anomalía (*outlier* en inglés) es un dato significativamente distinto a los demás. Se puede considerar como una observación cuya desviación del conjunto de datos permite establecer la hipótesis de que su generación fue obtenida por un mecanismo distinto al principal en la modelación de un fenómeno.

![Outlier](../../recursos/2023-01/16-Detección-anomalias/outliers.gif)

> **Pregunta**: ¿Por qué debería preocuparme por estos valores?

Las anomalías contienen, por tanto, información sobre características anormales de las entidades y esquemas que impactan el proceso generativo de los datos. Reconocer estas observaciones permite:
- Desde el punto de vista teórico, mejorar el entendimiento de los problemas modelados.
- Desde el punto de vista práctico, mejorar procesos de adquisición de datos y precisión de modelos.


Antes de continuar, es necesario hacer una distinción entre términos:

- **Detección de Outliers**: detectamos anomalías sobre los datos que estamos analizando o sobre los datos de entrenamiento del modelo.

- **Novelty Detection**: cuando detectamos outliers sobre **datos nuevos**.


## Tipos de outliers

- **Univariados**: aparecen en una sola característica de los datos.
- **Multivariados**: se detectan al combinar más de una característica de los datos.
   

## Origen de los outliers


> **Pregunta ❓**: ¿Cuáles son las posibles fuentes de outliers?


- Errores al transcribir los datos.
- Errores de medición.
- Errores experimentales.
- Errores de preprocesamiento.
- Al extraer o mezclar datos que no son compatibles entre sí.
- Naturales.

## ¿Qué hacemos después de detectar una anomalía?

Detectar una anomalía no implica automáticamente eliminarla. Antes de intervenir, conviene distinguir entre:

- un error de captura o transcripción,
- una medición defectuosa,
- un caso raro pero válido,
- una observación que debe escalarse al dominio de negocio.

Algunas acciones posibles son:

- **Eliminar** registros cuando existe evidencia de error irrecuperable.
- **Corregir** valores si hay una fuente confiable para hacerlo.
- **Winsorizar o capar** valores extremos cuando el objetivo es robustecer un modelo.
- **Imputar** si el tratamiento convierte el dato en faltante.
- **Separar y analizar** casos raros pero válidos, porque muchas veces contienen la información más interesante del problema.
- **Derivar al negocio o al experto de dominio** cuando la decisión no puede tomarse solo con criterios estadísticos.

En esta clase nos enfocaremos en **detectar** y **visualizar** anomalías, pero la decisión de qué hacer con ellas siempre depende del contexto.

## Dataset de Hoy: Wine Quality 🍷

<div style="text-align: center;">
    <img src="../../recursos/2023-01/16-Detección-anomalias/wine.jpg" style="width: 50%;">
</div>

<center> Fuente de la imagen: https://www.baysidegroup.com.au/blog/could-native-grapes-be-the-future-of-australian-wine/</center>

Wine Quality es un dataset de características que describen vinos portugueses de la variedad "Vinho Verde".

Atributos:

    1 - fixed acidity
    2 - volatile acidity
    3 - citric acid
    4 - residual sugar
    5 - chlorides
    6 - free sulfur dioxide
    7 - total sulfur dioxide
    8 - density
    9 - pH
    10 - sulphates
    11 - alcohol
    
Output: Calidad subjetiva del vino.

    12 - quality (score between 0 and 10)

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv(
    "../../recursos/2023-01/16-Detección-anomalias/wineQualityReds.csv", index_col=0
)
df.head()

In [ ]:
df.describe()

In [ ]:
for col in df:
    fig = px.histogram(df, x=col, marginal="box", title="Histogram of " + col)
    fig.show()

## Métodos de manejo de anomalías univariadas


#### Desviación estándar

Si se estima que la variable a estudiar se distribuye de manera normal, entonces el 95% de los datos se encuentra a 2 desviaciones estándar de la media, mientras que el 99.7% se encuentra dentro de 3 desviaciones estándar. Basándose en esto, se puede considerar cualquier punto fuera de 3 desviaciones estándar de la media como candidato a anomalía. Una forma más flexible de estimar anomalías usando el principio de normalidad es por medio de z-scores.


$$\text{z-score} = \frac{x_i - \overline{x}}{s}$$


<div style="text-align: center;">
    <img src="../../recursos/2023-01/16-Detección-anomalias/norm.png" style="width: 75%;">
</div>

In [ ]:
import plotly.figure_factory as ff

# Crear distplot con curva normal ajustada
fig = ff.create_distplot(
    [df["fixed.acidity"].values], ["fixed.acidity"], curve_type="normal", show_rug=False
)
fig.show()

In [ ]:
from scipy.stats import zscore

z_scores = zscore(df["fixed.acidity"])
z_scores

In [ ]:
# Agregamos los z-scores al dataframe
df["fixed.acidity_zscores"] = z_scores
df

In [ ]:
df["fixed.acidity_zscores"].abs() > 3

In [ ]:
# Agregamos un booleano por cada fila que indica si la observación está
# a 3 o más desviaciones estándar.
df["fixed.acidity_outlier"] = df["fixed.acidity_zscores"].abs() > 3

df[["fixed.acidity", "fixed.acidity_zscores", "fixed.acidity_outlier"]].head(10)

In [ ]:
df.loc[
    df["fixed.acidity_outlier"],  # condición: fixed.acidity_outlier == True
    ["fixed.acidity", "fixed.acidity_zscores", "fixed.acidity_outlier"],
]

In [ ]:
fig = px.histogram(df, x="fixed.acidity", color="fixed.acidity_outlier")
fig.update_layout(showlegend=False)


---
**✏️ Ejercicio — Z-score:**

Aplica el método de z-score a la columna `volatile.acidity`.

1. ¿Cuántos outliers detectas con umbral 3?
2. ¿Cambia el resultado si usas umbral 2? ¿Qué umbral te parece más adecuado para esta variable?
3. Grafica el histograma coloreado por outlier igual que se hizo con `fixed.acidity`.


In [ ]:
# Espacio para el ejercicio


Si se va a estudiar una columna con outliers mediante este método, conviene hacer un **test de normalidad**. Si la variable no cumple con esa hipótesis, una alternativa es aplicar transformaciones como **Box-Cox** o **Yeo-Johnson** para acercarla a un comportamiento más simétrico.

Si el supuesto de normalidad no es razonable, también conviene cambiar de criterio: en ese caso, el **rango intercuartílico (IQR)** suele ser una alternativa más robusta para detectar valores extremos, porque no depende tanto de la media ni de la desviación estándar.

#### IQR: Rango intercuartílico

El rango intercuartílico (**IQR**) se utiliza para medir la dispersión de los datos.
Para obtenerlo:

Los datos se separan en 4 grupos de (casi) igual tamaño.
El IQR se calcula como la diferencia entre el primer cuartil $Q1$ (25%) y el tercero $Q3$ (75%): $IQR = Q3 - Q1$.

<div align='center'>
    <img src="https://miro.medium.com/max/9000/1*2c21SkzJMf3frPXPAR_gZA.png" style="width: 50%;">
    <p>
        Fuente:
        <a href='https://www.kdnuggets.com/2019/11/understanding-boxplots.html'>Understanding Boxplots
        </a>
    </p>
</div>

In [ ]:
df["total.sulfur.dioxide"].describe()

In [ ]:
px.histogram(df, "total.sulfur.dioxide", marginal="box")

In [ ]:
desc = df["total.sulfur.dioxide"].describe()
desc

In [ ]:
iqr = desc["75%"] - desc["25%"]
iqr


Se consideran **candidatos a outlier** los valores que caen fuera de los siguientes límites:

$$\text{cota inferior} = Q1 - 1.5 \times IQR \qquad \text{cota superior} = Q3 + 1.5 \times IQR$$

Cualquier observación por debajo de la cota inferior o por sobre la cota superior queda fuera del rango "esperado" y se etiqueta como outlier.


In [ ]:
cota_inf = desc["25%"] - iqr * 1.5
cota_inf

In [ ]:
cota_sup = desc["75%"] + iqr * 1.5
cota_sup

In [ ]:
df["total.sulfur.dioxide_outlier"] = (df["total.sulfur.dioxide"] > cota_sup) | (
    df["total.sulfur.dioxide"] < cota_inf
)

In [ ]:
import plotly.express as px

fig1 = px.box(df, x="total.sulfur.dioxide", height=200)
fig1.show()

fig2 = px.histogram(
    df,
    x="total.sulfur.dioxide",
    color="total.sulfur.dioxide_outlier",
    title="total.sulfur.dioxide<br><sup>Rojo = outlier</sup>",
)
fig2.update_layout(showlegend=False)
fig2.show()


---
**✏️ Ejercicio — IQR:**

Aplica el método IQR a la columna `chlorides`.

1. Calcula las cotas inferior y superior.
2. ¿Cuántos outliers detectas?
3. Compara con el z-score de la misma columna. ¿Coinciden todos los outliers? ¿Por qué podrían diferir?


In [ ]:
# Espacio para el ejercicio


En el gráfico de caja, vemos que los outliers están sobre y bajo las líneas rectas. Cada una representa $Q1 - 1.5 \times IQR$ (línea inferior) y $Q3 + 1.5 \times IQR$ (línea superior). Los valores dentro de la caja corresponden al IQR, y la línea central es la mediana de los datos.


> **¿Por qué 1.5 y no otro número?**

El valor 1.5 fue propuesto por John Tukey en 1977. Para una distribución normal, los límites $Q1 - 1.5 \times IQR$ y $Q3 + 1.5 \times IQR$ excluyen aproximadamente el **0.7 % más extremo** de los datos, lo que empíricamente resulta en un umbral que equilibra sensibilidad y especificidad en distribuciones simétricas.

Usar **3 × IQR** en cambio es más conservador: solo etiqueta como outlier el ~0.01 % más extremo, y es útil cuando los datos tienen colas pesadas (distribuciones asimétricas o con muchos valores extremos legítimos).

<div style="text-align: center;">
    <img src="../../recursos/2023-01/16-Detección-anomalias/iqr_origen.png" style="width: 50%;">
</div>



## Reducción de dimensionalidad para la visualización

Los métodos univariados (z-score, IQR) detectan anomalías variable por variable, pero un dato puede ser completamente normal en cada columna por separado y aun así ser inusual cuando se considera la **combinación** de todas sus características.

Para poder ver esto, necesitamos una forma de **representar todas las variables al mismo tiempo** en un espacio que podamos graficar (2D o 3D). Ahí es donde entran los métodos de reducción de dimensionalidad para visualización.

El dataset tiene 11 variables. Representar cada observación en 11 dimensiones es imposible de visualizar. Estos métodos transforman cada observación (vector de 11 dimensiones) en un punto en 2D, intentando preservar la mayor cantidad de información estructural posible.



In [ ]:
df.head()



Por lo tanto, sería de mucha utilidad contar con mecanismos para reducir la cantidad de características. Existen 2 enfoques:

- **Selección de Atributos Relevantes**
- **Reducción de Dimensionalidad.**

Ambos serán vistos con mayor profundidad más adelante.

Sin embargo, existen un par de métodos de reducción de dimensionalidad que nos pueden ser particularmente útiles para la tarea que estamos resolviendo en este momento (detección de outliers) y, en general, para crear una idea de cómo se distribuyen los datos al considerar todas las características.

Estos son los **métodos de reducción de dimensionalidad para visualización**. Son técnicas que nos permiten representar cada observación (un vector de alta dimensionalidad) en un vector de dos dimensiones (mucho más sencillo para graficar).

Veremos **cuatro métodos** en esta clase:

1. **PCA** — método lineal. Sirve como referencia rápida e interpretable.
2. **t-SNE** — no lineal. Enfatiza estructura local.
3. **UMAP** — no lineal. Equilibra local y global, más rápido que t-SNE.
4. **PaCMAP** — no lineal. Diseñado para ser estable entre ejecuciones.

Empezamos con PCA como punto de partida, luego comparamos con los métodos no lineales.

**Paréntesis: Manifold learning**

> Manifold learning es un enfoque para la reducción de dimensionalidad no lineal. Los algoritmos para esta tarea se basan en la idea de que la dimensionalidad de muchos conjuntos de datos es solo artificialmente alta.


In [ ]:
features = df.drop(
    columns=[
        "quality",
        "fixed.acidity_zscores",
        "fixed.acidity_outlier",
        "total.sulfur.dioxide_outlier",
    ]
)
quality = df.loc[:, ["quality"]]

features.head(3)


### PCA (Análisis de Componentes Principales)

PCA es el punto de partida: un método **lineal** que busca las direcciones de máxima varianza en los datos y proyecta cada observación sobre ellas.

Usamos PCA como **referencia** porque:

- es rápido y completamente reproducible (no tiene aleatoriedad),
- permite cuantificar cuánta varianza del dataset capturan las 2 dimensiones (varianza explicada),
- permite interpretar qué variables originales dirigen cada componente principal (loadings),
- sirve para evaluar si los métodos no lineales aportan información visual que PCA no logra capturar.

> **Nota**: PCA escala los datos linealmente, por lo que cualquier estructura no lineal en los datos quedará colapsada.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

# Escalamos una sola vez y reutilizamos para PCA, t-SNE y UMAP.
scaled_features = MinMaxScaler().fit_transform(features)

pca = PCA(n_components=2, random_state=42)
wine_features_pca_embedded = pca.fit_transform(scaled_features)

df["x_pca"] = wine_features_pca_embedded[:, 0]
df["y_pca"] = wine_features_pca_embedded[:, 1]

print(
    "Varianza explicada acumulada (2 componentes):", pca.explained_variance_ratio_.sum()
)
px.scatter(df, x="x_pca", y="y_pca", color="quality")

#### Direcciones de las features en PCA (loadings)

Una ventaja de PCA es que, además de proyectar observaciones, permite visualizar hacia dónde apuntan las variables originales en el plano de componentes principales.

En el siguiente gráfico, cada flecha representa la dirección de una feature en el espacio PCA 2D.

In [ ]:
import plotly.graph_objects as go

loadings = pca.components_.T  # shape: (n_features, 2)
scale = 4.0

fig_loadings = go.Figure()

# Puntos proyectados en PCA para referencia visual
fig_loadings.add_trace(
    go.Scatter(
        x=df["x_pca"],
        y=df["y_pca"],
        mode="markers",
        marker=dict(size=5, opacity=0.3, color="lightgray"),
        name="Observaciones",
    )
)

# Flechas de las features (loadings)
for i, feature_name in enumerate(features.columns):
    x_end = loadings[i, 0] * scale
    y_end = loadings[i, 1] * scale

    fig_loadings.add_trace(
        go.Scatter(
            x=[0, x_end],
            y=[0, y_end],
            mode="lines+markers+text",
            marker=dict(size=[0, 8]),
            line=dict(width=2),
            text=["", feature_name],
            textposition="top center",
            name=feature_name,
            showlegend=False,
        )
    )

fig_loadings.update_layout(
    title="PCA: direcciones de las features (loadings)",
    xaxis_title="Componente principal 1",
    yaxis_title="Componente principal 2",
    width=800,
    height=550,
    template="plotly_white",
)
fig_loadings.show()


### t-distributed Stochastic Neighbor Embedding (t-SNE)

Habiendo visto PCA como referencia lineal, pasamos al primer método **no lineal**: t-SNE. A diferencia de PCA, t-SNE no busca maximizar varianza sino **conservar vecindarios**: observaciones similares quedan proyectadas cerca, observaciones distintas quedan lejos.

Esto lo hace especialmente útil para revelar clusters y subestructuras que PCA —al ser lineal— aplanaría. Sin embargo, t-SNE tiene importantes limitaciones:

- **No preserva distancias globales**: dos clusters alejados en la proyección no necesariamente lo están en el espacio original.
- **Es estocástico**: distintas ejecuciones con distinta semilla pueden producir proyecciones muy diferentes.
- **Es computacionalmente caro**: puede tomar horas con más de un millón de datos.
- **No permite transformar datos nuevos**: una vez entrenado, no hay un `.transform` para puntos nuevos.

> **Nota**: Es importante escalar todos los datos a la misma escala antes de aplicar t-SNE.

#### Hiperparámetros clave

| Parámetro | Descripción | Valor típico |
| --- | --- | --- |
| `perplexity` | Controla el tamaño efectivo del vecindario de cada punto. Valores bajos capturan estructura muy local; valores altos la suavizan. **Tiene gran impacto visual**: conviene probar 5, 30 y 50 y comparar. | 5–50 (default: 30) |
| `n_components` | Dimensión del espacio de salida. Para visualización, 2. | 2 |
| `learning_rate` | Tasa de aprendizaje del gradiente. Valores muy bajos producen clusters comprimidos; muy altos, proyecciones ruidosas. | 10–1000 (default: `"auto"`) |
| `n_iter` | Número de iteraciones de optimización. Más iteraciones = más calidad pero más lento. | ≥ 250 (default: 1000) |
| `random_state` | Semilla para reproducibilidad. Siempre fijarla al comparar proyecciones. | cualquier entero |

<div style="text-align: center;">
    <img src="https://miro.medium.com/max/800/1*lKLB_1aghhnxQjMQziEyGQ.gif" style="width: 50%;">
</div>

<center>Fuente: https://www.oreilly.com/content/an-illustrated-introduction-to-the-t-sne-algorithm/ — Ver también: <a href="https://projector.tensorflow.org/">TensorFlow Projector</a></center>


In [ ]:
from sklearn.manifold import TSNE

# Reutilizamos los datos ya escalados en el baseline PCA.
# Creamos una instancia de t-SNE.
tsne = TSNE(n_components=2, random_state=42)

# Transformamos los datos.
wine_features_tsne_embedded = tsne.fit_transform(scaled_features)
wine_features_tsne_embedded

In [ ]:
# Noten que guardamos la transformación en df y no en features.
df["x_tsne"] = wine_features_tsne_embedded[:, 0]
df["y_tsne"] = wine_features_tsne_embedded[:, 1]


### ¿Qué explica la estructura de la proyección?

Proyectar los datos en 2D es un primer paso. El siguiente es **interpretar** esa proyección coloreando los puntos por distintas variables del dataset.

Si una variable tiene valores similares en regiones compactas del plano, esa variable aporta estructura a los datos. Si el color se mezcla aleatoriamente, la variable no explica los clusters visibles.

A continuación coloreamos la proyección t-SNE por seis variables: variables continuas (alcohol, pH, azúcar residual, acidez fija, calidad) y el indicador de outlier univariado calculado antes.


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

vars_continuas = ["alcohol", "pH", "residual.sugar", "fixed.acidity", "quality"]
var_categorica = "fixed.acidity_outlier"
all_vars = vars_continuas + [var_categorica]
subplot_titles = all_vars

fig_tsne_grid = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.06,
    vertical_spacing=0.15,
)

positions = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)]
palette = px.colors.qualitative.Plotly

for (r, c), var in zip(positions, all_vars):
    if var in vars_continuas:
        fig_tsne_grid.add_trace(
            go.Scatter(
                x=df["x_tsne"],
                y=df["y_tsne"],
                mode="markers",
                marker=dict(
                    size=4, color=df[var], colorscale="Viridis", showscale=True
                ),
                name=var,
                showlegend=False,
            ),
            row=r,
            col=c,
        )
    else:
        for i, cat in enumerate(sorted(df[var].unique())):
            mask = df[var] == cat
            fig_tsne_grid.add_trace(
                go.Scatter(
                    x=df.loc[mask, "x_tsne"],
                    y=df.loc[mask, "y_tsne"],
                    mode="markers",
                    marker=dict(size=4, color=palette[i % len(palette)]),
                    name=f"outlier={cat}",
                    legendgroup=var,
                    showlegend=True,
                ),
                row=r,
                col=c,
            )

fig_tsne_grid.update_layout(
    height=680,
    title_text="t-SNE: ¿qué variables explican la estructura de la proyección?",
    template="plotly_white",
)
fig_tsne_grid.show()



### Uniform Manifold Approximation and Projection (UMAP)

UMAP es un método relativamente reciente de reducción de dimensionalidad no lineal. A diferencia de t-SNE, puede usarse tanto para **visualización** como para **reducción de dimensionalidad como preprocesamiento** de modelos predictivos.

Sus principales ventajas sobre t-SNE son:

- **Más rápido** y escala mejor con el tamaño del dataset.
- **Preserva mejor la estructura global** (distancias entre grupos), no solo la local.
- **Permite transformar datos nuevos** con `.transform()` una vez entrenado: es útil para novelty detection en producción.

> **Nota metodológica**: UMAP es muy útil para **visualización exploratoria**, pero la proyección 2D no reemplaza al espacio original. Si luego queremos clusterizar o detectar anomalías sobre la proyección, conviene interpretar ese resultado como una ayuda visual y validarlo contra las variables originales.

#### Hiperparámetros clave

| Parámetro | Descripción | Valor típico |
| --- | --- | --- |
| `n_neighbors` | Número de vecinos considerados para la estructura local. Valores pequeños enfatizan detalles finos; valores grandes suavizan y capturan más estructura global. | 5–50 (default: 15) |
| `min_dist` | Distancia mínima entre puntos en el espacio proyectado. Valores pequeños producen clusters más compactos; valores grandes los "aplastan". | 0.0–0.99 (default: 0.1) |
| `n_components` | Dimensión de salida. Para visualización, 2. | 2 |
| `metric` | Métrica de distancia en el espacio original. Para datos numéricos continuos, `"euclidean"` es un buen punto de partida. | `"euclidean"` |
| `random_state` | Semilla de reproducibilidad. | cualquier entero |


In [ ]:
import umap

# Reutilizamos los datos escalados en el baseline PCA.
umap_reducer = umap.UMAP(n_components=2, random_state=42)

wine_features_umap_embedded = umap_reducer.fit_transform(scaled_features)
wine_features_umap_embedded

In [ ]:
# Noten que guardamos la transformación en df y no en features.
df["x_umap"] = wine_features_umap_embedded[:, 0]
df["y_umap"] = wine_features_umap_embedded[:, 1]


### PaCMAP (Pairwise Controlled Manifold Approximation)

PaCMAP es un método de reducción de dimensionalidad no lineal diseñado para **equilibrar estructura local y global** de manera más estable que t-SNE o UMAP, siendo menos sensible a la elección de hiperparámetros.

Es una buena opción cuando se quiere una proyección **reproducible y comparativamente estable** entre distintas ejecuciones y configuraciones, especialmente para validar si los clusters encontrados con UMAP o t-SNE se mantienen.

#### Hiperparámetros clave

| Parámetro | Descripción | Valor típico |
| --- | --- | --- |
| `n_components` | Dimensión de salida. | 2 |
| `n_neighbors` | Vecinos para estructura local. Similar al de UMAP. | 10 (default) |
| `MN_ratio` | Controla el balance entre pares cercanos y pares a distancia media. Valores altos priorizan estructura global. | 0.5 (default) |
| `FP_ratio` | Controla la repulsión con puntos lejanos. | 2.0 (default) |
| `random_state` | Semilla de reproducibilidad. | cualquier entero |


In [ ]:
import pacmap

# Reutilizamos los datos escalados en el baseline PCA.
pacmap_reducer = pacmap.PaCMAP(n_components=2, random_state=42)
wine_features_pacmap_embedded = pacmap_reducer.fit_transform(scaled_features)

df["x_pacmap"] = wine_features_pacmap_embedded[:, 0]
df["y_pacmap"] = wine_features_pacmap_embedded[:, 1]

wine_features_pacmap_embedded

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

projections = {
    "PCA": ("x_pca", "y_pca"),
    "t-SNE": ("x_tsne", "y_tsne"),
    "UMAP": ("x_umap", "y_umap"),
    "PaCMAP": ("x_pacmap", "y_pacmap"),
}

outlier_col = "total.sulfur.dioxide_outlier"
quality_col = "quality"

color_map = {False: "steelblue", True: "firebrick"}
label_map = {False: "Normal", True: "Outlier IQR"}

fig = make_subplots(
    rows=2,
    cols=4,
    subplot_titles=list(projections.keys()) * 2,
    row_titles=["Outliers IQR (total sulfur dioxide)", "Calidad del vino"],
    horizontal_spacing=0.04,
    vertical_spacing=0.18,
)

for col_idx, (method, (x_col, y_col)) in enumerate(projections.items(), start=1):
    # --- Fila 1: color por calidad ---
    fig.add_trace(
        go.Scatter(
            x=df[x_col],
            y=df[y_col],
            mode="markers",
            marker=dict(
                size=4,
                color=df[quality_col],
                colorscale="RdYlGn",
                cmin=df[quality_col].min(),
                cmax=df[quality_col].max(),
                showscale=(col_idx == 4),
                colorbar=dict(title="Calidad", len=0.42, y=0.12, x=1.02),
                opacity=0.75,
            ),
            showlegend=False,
        ),
        row=1,
        col=col_idx,
    )

    # --- Fila 2: color por outlier IQR ---
    for is_outlier in [False, True]:
        mask = df[outlier_col] == is_outlier
        fig.add_trace(
            go.Scatter(
                x=df.loc[mask, x_col],
                y=df.loc[mask, y_col],
                mode="markers",
                marker=dict(size=4, color=color_map[is_outlier], opacity=0.75),
                name=label_map[is_outlier],
                legendgroup="outlier",
                showlegend=(col_idx == 1),
            ),
            row=2,
            col=col_idx,
        )

fig.update_layout(
    height=720,
    title_text="Comparación de proyecciones: calidad del vino (fila 1) y outliers IQR (fila 2)",
)
fig.show()



### Resumen de métodos de reducción de dimensionalidad

| Método | Tipo | Qué preserva mejor | Parámetros clave | Ventajas | Limitaciones |
| --- | --- | --- | --- | --- | --- |
| **PCA** | Lineal | Varianza global lineal | `n_components` | Rápido, reproducible, interpretable (loadings), permite `transform` | No captura estructura no lineal |
| **t-SNE** | No lineal | Vecindarios locales | `perplexity`, `learning_rate`, `n_iter` | Muy visual para grupos locales | Distancias globales no confiables, sin `transform`, lento |
| **UMAP** | No lineal | Local + parte de la global | `n_neighbors`, `min_dist`, `metric` | Rápido, permite `transform`, preserva algo de estructura global | Puede distorsionar relaciones según configuración |
| **PaCMAP** | No lineal | Equilibrio local-global | `n_neighbors`, `MN_ratio`, `FP_ratio` | Estable entre ejecuciones, menos sensible a hiperparámetros | Menos estándar en flujos productivos |



> **Reflexión:** Los métodos univariados (z-score, IQR) detectan extremos en una sola variable a la vez. ¿Qué pasa con un vino cuyo pH, alcohol y densidad son individualmente normales pero cuya **combinación** nunca aparece en el resto del dataset?

Ese tipo de anomalía es invisible para los detectores univariados. Necesitamos métodos que consideren **todas las variables simultáneamente**: los detectores multivariados.


## Métodos de manejo de outliers multivariados


Este tipo de métodos permite encontrar outliers considerando no solo un atributo en particular, sino todos los atributos al mismo tiempo.

### DBSCAN

DBSCAN es un algoritmo de clustering basado en densidad. Su funcionamiento se basa en clasificar los datos en tres categorías:

- **Core point**: es un punto que contiene un número mínimo de vecinos en un vecindario (esfera de radio ɛ).

- **Border point**: es un punto que no es core, pero que es alcanzable por un core point. Es decir, existe un camino entre el core point y este.

- **Outlier**: es un punto que no es core point y que, a la vez, no tiene un camino desde un core point o un border point.

<div style="text-align: center;">
    <img src="../../recursos/2023-01/16-Detección-anomalias/dbscan.png" style="width: 50%;">
</div>

In [ ]:
from sklearn.cluster import DBSCAN

# El parámetro eps indica el tamaño de la esfera y, por tanto, la cantidad de outliers.
# Mientras menor sea el tamaño de eps, mayor la cantidad de outliers.
clustering = DBSCAN(eps=0.5, min_samples=3).fit(scaled_features)

db_scan_labels = clustering.labels_

db_scan_labels

In [ ]:
db_scan_labels.max()

In [ ]:
# Nota: Los labels de los outliers son -1
df[db_scan_labels == -1]

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

projections = {
    "PCA": ("x_pca", "y_pca"),
    "t-SNE": ("x_tsne", "y_tsne"),
    "UMAP": ("x_umap", "y_umap"),
    "PaCMAP": ("x_pacmap", "y_pacmap"),
}

is_outlier_mask = db_scan_labels == -1

fig_dbscan = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=list(projections.keys()),
    horizontal_spacing=0.04,
)

for col_idx, (proj_name, (x_col, y_col)) in enumerate(projections.items(), start=1):
    for val, color, leg_name in [
        (False, "steelblue", "Normal"),
        (True, "firebrick", "Outlier DBSCAN"),
    ]:
        mask = is_outlier_mask == val
        fig_dbscan.add_trace(
            go.Scatter(
                x=df.loc[mask, x_col],
                y=df.loc[mask, y_col],
                mode="markers",
                marker=dict(size=4, color=color, opacity=0.75),
                name=leg_name,
                legendgroup="outlier",
                showlegend=(col_idx == 1),
            ),
            row=1,
            col=col_idx,
        )

fig_dbscan.update_layout(
    height=400,
    title_text="DBSCAN: outliers en las 4 proyecciones",
    template="plotly_white",
)
fig_dbscan.show()



## Detección de anomalías con PyOD

PyOD entrega una **API unificada** para múltiples detectores (`fit`, `decision_scores_`, `decision_function`, `predict`), lo que facilita comparar métodos bajo el mismo protocolo sin reescribir el flujo.

### Métodos disponibles

| Método (PyOD) | Tipo | Idea principal | Parámetros clave | Cuándo usarlo |
| --- | --- | --- | --- | --- |
| `ECOD` | Probabilístico | Usa distribuciones empíricas por feature para puntuar rareza | `contamination` | Baseline robusto y rápido en datos tabulares |
| `IForest` | Ensamble | Aísla puntos anómalos con particiones aleatorias | `contamination`, `n_estimators`, `max_samples`, `random_state` | Baseline general en media/alta dimensión |
| `LOF` | Local | Compara densidad local con vecinos | `contamination`, `n_neighbors` | Cuando la anomalía depende del vecindario local |
| `HBOS` | Histograma | Evalúa rareza por histogramas univariados | `contamination`, `n_bins` | Cuando se necesita velocidad e interpretación simple |
| `COPOD` | Copulas | Modela colas y dependencias con enfoque robusto | `contamination` | Buena opción tabular sin mucha calibración |
| `KNN` | Distancia | Puntúa por distancia a vecinos cercanos | `contamination`, `n_neighbors`, `method` | Referencia intuitiva de proximidad |
| `PCA` | Proyección | Usa residuo fuera de subespacios principales | `contamination`, `n_components` | Datos con estructura aproximadamente lineal |

En esta clase aplicaremos **ECOD**, **IForest** y **LOF** como núcleo, y dejaremos el resto como exploración guiada.


### Detectores PyOD: ECOD, IForest, LOF, HBOS y KNN

Trabajaremos con cinco detectores complementarios, cada uno con un principio de funcionamiento distinto:

#### **ECOD** *(Empirical Cumulative Distribution)*

Estima la probabilidad de cada observación usando las distribuciones empíricas marginales de cada variable. Un punto es anómalo si cae en las colas de muchas dimensiones simultáneamente. No asume ninguna distribución paramétrica.


#### **IForest** *(Isolation Forest)*

Construye árboles de decisión aleatorios y mide cuántos cortes se necesitan para aislar cada punto. Los outliers, al estar alejados de la masa de datos, se aíslan en pocos cortes.


### **LOF** *(Local Outlier Factor)*

Compara la densidad local de cada punto con la de sus vecinos. Un punto es anómalo si su vecindario es mucho menos denso que el de sus propios vecinos, capturando anomalías relativas al contexto local.


### **HBOS** *(Histogram-Based Outlier Score)*

Construye un histograma independiente por cada variable y multiplica las frecuencias. Los puntos que caen en bins poco frecuentes en varias dimensiones reciben scores altos. Es muy rápido al asumir independencia entre variables.

### **KNN** *(k-Nearest Neighbors)*

Ccalcula la distancia de cada punto a su k-ésimo vecino más cercano. Cuanto más lejos estén sus vecinos, mayor es el score de anomalía. Es simple e intuitivo, pero sensible a la escala y al valor de k.


Todos entregan un score continuo de anomalía y una etiqueta binaria, lo que permite comparar umbrales, estabilidad y concordancia entre métodos en el mismo flujo.

In [ ]:
from pyod.models.ecod import ECOD
from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.hbos import HBOS
from pyod.models.knn import KNN

detectors = {
    "ECOD": ECOD(contamination=0.01),
    "IForest": IForest(contamination=0.01, random_state=42),
    "LOF": LOF(contamination=0.01),
    "HBOS": HBOS(contamination=0.01),
    "KNN": KNN(contamination=0.01),
}

detector_scores = {}
detector_labels = {}

for name, clf in detectors.items():
    clf.fit(scaled_features)
    detector_scores[name] = pd.Series(clf.decision_scores_, index=df.index)
    detector_labels[name] = pd.Series(
        clf.labels_, index=df.index
    )  # 1=outlier, 0=inlier

# Mantener una referencia explícita para la sección de novelty/umbral
pyod_iforest = detectors["IForest"]


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

projections = {
    "PCA": ("x_pca", "y_pca"),
    "t-SNE": ("x_tsne", "y_tsne"),
    "UMAP": ("x_umap", "y_umap"),
    "PaCMAP": ("x_pacmap", "y_pacmap"),
}

detector_names = ["ECOD", "IForest", "LOF", "HBOS", "KNN"]

subplot_titles = [f"{det} — {proj}" for det in detector_names for proj in projections]

fig_pyod = make_subplots(
    rows=5,
    cols=4,
    subplot_titles=subplot_titles,
    row_titles=detector_names,
    horizontal_spacing=0.04,
    vertical_spacing=0.08,
)

for row_idx, det_name in enumerate(detector_names, start=1):
    labels = detector_labels[det_name]
    for col_idx, (proj_name, (x_col, y_col)) in enumerate(projections.items(), start=1):
        for val, color, leg_name in [
            (0, "steelblue", "Normal"),
            (1, "firebrick", "Outlier"),
        ]:
            mask = labels == val
            fig_pyod.add_trace(
                go.Scatter(
                    x=df.loc[mask, x_col],
                    y=df.loc[mask, y_col],
                    mode="markers",
                    marker=dict(size=4, color=color, opacity=0.75),
                    name=leg_name,
                    legendgroup="pyod_label",
                    showlegend=(row_idx == 1 and col_idx == 1),
                ),
                row=row_idx,
                col=col_idx,
            )

fig_pyod.update_layout(
    height=1400,
    title_text="Comparación de detectores PyOD (ECOD, IForest, LOF, HBOS, KNN) en las 4 proyecciones",
    legend=dict(orientation="h", y=1.03, x=0.0),
)
fig_pyod.show()


#### Novelty Detection con PyOD (IForest)

Con el detector entrenado, podemos clasificar una nueva observación (fuera del conjunto de entrenamiento) como outlier o no usando `.predict`.

In [ ]:
# Tomamos una observación real del dataset para usarla como referencia "normal".
# scaled_features ya está normalizado entre 0 y 1 con MinMaxScaler.
row_normal = scaled_features[100, :].copy()
print("Predicción para muestra normal (fila 100):", pyod_iforest.predict([row_normal]))


In [ ]:
# Construimos un vino "anómalo" de forma realista.
# Como los datos están escalados (0=mínimo del dataset, 1=máximo), podemos
# simular combinaciones extremas pero físicamente plausibles:
#   - acidez volátil al máximo (muy vinagre)
#   - azúcar residual al máximo (muy dulce)
#   - alcohol al mínimo
#   - sulfatos al máximo
# Esa combinación nunca aparece junta en el dataset.
row_anomalo = scaled_features[100, :].copy()
row_anomalo[features.columns.get_loc("volatile.acidity")] = 1.0  # máx acidez volátil
row_anomalo[features.columns.get_loc("residual.sugar")] = 1.0  # máx azúcar
row_anomalo[features.columns.get_loc("alcohol")] = 0.0  # mín alcohol
row_anomalo[features.columns.get_loc("sulphates")] = 1.0  # máx sulfatos

print(
    "Predicción para vino con características extremas:",
    pyod_iforest.predict([row_anomalo]),
)
# 0 = inlier, 1 = outlier (convenio PyOD)


## Cierre: score, etiqueta y validación

### 1) Score de anomalía vs etiqueta binaria

Muchos detectores entregan:
- un **score continuo** de anormalidad (qué tan raro es un punto)
- una **etiqueta binaria** (outlier/inlier).
  
La etiqueta depende de una regla de corte, por lo que dos umbrales distintos pueden producir etiquetas distintas sobre los mismos scores.

### 2) Contamination y umbral

`contamination` representa la fracción esperada de outliers. En la práctica, puede usarse para definir un umbral inicial sobre scores, pero no debería tratarse como verdad absoluta.

### 3) ¿Cómo validar sin ground truth?

Si no tenemos etiquetas reales, conviene combinar evidencia:

- estabilidad del detector ante cambios de hiperparámetros,
- consistencia entre métodos distintos,
- revisión de casos extremos con conocimiento de dominio,
- impacto real en métricas de negocio o calidad de datos.

In [ ]:
# Ejemplo simple: pasar de score continuo a etiqueta por umbral.
scores_iforest_pyod = detector_scores["IForest"]

# En PyOD, scores más altos suelen indicar mayor anormalidad.
umbral_95pct = scores_iforest_pyod.quantile(0.95)
etiqueta_por_umbral = (scores_iforest_pyod >= umbral_95pct).astype(int)  # 1 = outlier

pd.DataFrame(
    {
        "score_iforest_pyod": scores_iforest_pyod,
        "etiqueta_umbral_95pct": etiqueta_por_umbral,
    }
).head()